In [10]:
import cv2
import numpy as np
from skimage import morphology, filters, segmentation, measure
from scipy import ndimage as ndi

def maskcreator(img_path, save_path=None):
    
    # load full image
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
    # apply contrast 
    clahe = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8,8))
    clahe_image = clahe.apply(image)
    thresh = filters.threshold_otsu(clahe_image)
    bimask = clahe_image < thresh

    #apply watershed
    dist = ndi.distance_transform_edt(bimask)
    maxima = morphology.local_maxima(dist)
    marker,_ = ndi.label(maxima)
    label = segmentation.watershed(-dist, marker, mask=bimask)

    #seperate mask
    mask = np.zeros_like(image, dtype=np.uint8)
    for region in measure.regionprops(label):
        if region.area > 80:
            for coord in region.coords:
                mask[coord[0],coord[1]] = 255

    if save_path:
        cv2.imwrite(save_path, mask)

    return mask

In [ ]:
import os
input_folder = "Non_covid_X-rays" #replace with dataset folder
output_folder = "ggo_masks" #replace with destination mask folder 
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith(('.png','.jpg','.jpeg')):
        img_path = os.path.join(input_folder, filename)
        save_path = os.path.join(output_folder, filename.replace('.jpg','.png').replace('.jpeg','.png'))
        mask = maskcreator(img_path, save_path=save_path)
